# 7-bob. 7-BOB. ZARARLI DASTURLARNI ANIQLASH I: STATIK TAHLIL
## Kengaytirilgan amaliy ish: MinHash yordamida zararli dastur variantlarini guruhlash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/07-bob/7-1_minhash-yordamida-zararli-dastur-variantlarini.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 7-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur uchta "oila" yaratadi va har bir oiladan uchta variant hosil qiladi. Har bir variantda baytlarning 2 foizi o'zgartiriladi va tasodifiy joyga 500 baytli yangi kod bo'lagi qo'shiladi. So'ngra fayllar SHA-256 va 128 ta xesh funksiyali MinHash yordamida taqqoslanadi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import hashlib
import numpy as np

rng = np.random.default_rng(7)

In [2]:
# 1. Sun'iy "fayllar": uchta oila va har bir oilaning o'zgartirilgan variantlari
def oila_asosi(uzunlik=20_000):
    return rng.integers(0, 256, uzunlik, dtype=np.uint8)

def variant(asos, ozgarish=0.02, qoshimcha=500):
    b = asos.copy()
    joy = rng.random(len(b)) < ozgarish                    # baytlarning 2 foizi o'zgaradi
    b[joy] = rng.integers(0, 256, joy.sum())
    kiritma = rng.integers(0, 256, qoshimcha, dtype=np.uint8)   # o'rtaga yangi kod bo'lagi qo'shiladi
    k = rng.integers(0, len(b))
    return np.concatenate([b[:k], kiritma, b[k:]]).tobytes()

asoslar = {nom: oila_asosi() for nom in ("A", "B", "C")}
fayllar = {f"{nom}{i}": variant(a) for nom, a in asoslar.items() for i in range(1, 4)}

In [3]:
# 2. MinHash: fayldagi 8 baytli bo'laklar to'plamining ixcham "barmoq izi"
K, N_XESH = 8, 128
tub = (1 << 61) - 1
a_k = rng.integers(1, tub, N_XESH, dtype=np.uint64)
b_k = rng.integers(0, tub, N_XESH, dtype=np.uint64)

def minhash(data):
    bolaklar = {int.from_bytes(hashlib.blake2b(data[i:i + K], digest_size=7).digest(), "big")
                for i in range(len(data) - K + 1)}
    x = np.fromiter(bolaklar, dtype=np.uint64)
    # har bir xesh funksiya uchun minimal qiymat (oddiylik uchun 64 bitli toshib ketish bilan)
    return np.array([((a * x + b) % np.uint64(tub)).min() for a, b in zip(a_k, b_k)])

imzolar = {nom: minhash(d) for nom, d in fayllar.items()}
oxshashlik = lambda m1, m2: (m1 == m2).mean()          # Jakkar o'xshashligining bahosi

print("SHA-256 (birinchi 12 belgi):")
for nom in ("A1", "A2", "B1"):
    print(f"  {nom}: {hashlib.sha256(fayllar[nom]).hexdigest()[:12]}")

nomlar = list(fayllar)
print("\nMinHash o'xshashligi:")
print("      " + "".join(f"{n:>6}" for n in nomlar))
for n1 in nomlar:
    print(f"{n1:>6}" + "".join(f"{oxshashlik(imzolar[n1], imzolar[n2]):6.2f}" for n2 in nomlar))

SHA-256 (birinchi 12 belgi):
  A1: a5da0a5e6f1c
  A2: 1fda1298245a
  B1: 198e2c04bef0

MinHash o'xshashligi:
          A1    A2    A3    B1    B2    B3    C1    C2    C3
    A1  1.00  0.54  0.54  0.00  0.00  0.00  0.00  0.00  0.00
    A2  0.54  1.00  0.50  0.00  0.00  0.00  0.00  0.00  0.00
    A3  0.54  0.50  1.00  0.00  0.00  0.00  0.00  0.00  0.00
    B1  0.00  0.00  0.00  1.00  0.54  0.56  0.00  0.00  0.00
    B2  0.00  0.00  0.00  0.54  1.00  0.54  0.00  0.00  0.00
    B3  0.00  0.00  0.00  0.56  0.54  1.00  0.00  0.00  0.00
    C1  0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.57  0.57
    C2  0.00  0.00  0.00  0.00  0.00  0.00  0.57  1.00  0.55
    C3  0.00  0.00  0.00  0.00  0.00  0.00  0.57  0.55  1.00


In [4]:
# 3. Yangi namuna kelganda: bazadagi eng o'xshash faylni topish
yangi = variant(asoslar["B"], ozgarish=0.05, qoshimcha=3000)
m = minhash(yangi)
eng = max(nomlar, key=lambda n: oxshashlik(m, imzolar[n]))
print(f"\nYangi namuna: eng o'xshashi {eng}, o'xshashlik {oxshashlik(m, imzolar[eng]):.2f}")


Yangi namuna: eng o'xshashi B1, o'xshashlik 0.41


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
SHA-256 (birinchi 12 belgi):
  A1: a5da0a5e6f1c
  A2: 1fda1298245a
  B1: 198e2c04bef0

MinHash o'xshashligi:
          A1    A2    A3    B1    B2    B3    C1    C2    C3
    A1  1.00  0.54  0.54  0.00  0.00  0.00  0.00  0.00  0.00
    A2  0.54  1.00  0.50  0.00  0.00  0.00  0.00  0.00  0.00
    A3  0.54  0.50  1.00  0.00  0.00  0.00  0.00  0.00  0.00
    B1  0.00  0.00  0.00  1.00  0.54  0.56  0.00  0.00  0.00
    B2  0.00  0.00  0.00  0.54  1.00  0.54  0.00  0.00  0.00
    B3  0.00  0.00  0.00  0.56  0.54  1.00  0.00  0.00  0.00
    C1  0.00  0.00  0.00  0.00  0.00  0.00  1.00  0.57  0.57
    C2  0.00  0.00  0.00  0.00  0.00  0.00  0.57  1.00  0.55
    C3  0.00  0.00  0.00  0.00  0.00  0.00  0.57  0.55  1.00

Yangi namuna: eng o'xshashi B1, o'xshashlik 0.41
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 7-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.